# 🧬 13 · Image Synthesis and Generative Models
### Not just recognizing cells, but generating new, realistic ones: shape spaces, variational autoencoders, and adversarial networks

**Part F · Segmenting, detecting, synthesizing** · Module 13 of 29 (core) · builds on 08, 09, 11

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xulabs/edu/blob/main/sci_img/tutorial/BiologyToBioimage/notebooks/13_image_synthesis_and_generative_models.ipynb)

| | |
|---|---|
| **Hardware target** | CPU |
| **Core frameworks** | NumPy, PyTorch, scikit-learn |
| **Prerequisites** | Module 08 (CNN basics), Module 09 (PCA/MDS, autoencoders), Module 11 (deep segmentation) |
| **Estimated time** | 110–140 minutes |
| **02-740 topics covered** | generative vs. descriptive models, diffeomorphic shape-space models, variational autoencoders (architecture, reparameterization, ELBO, latent perturbation), generative adversarial networks and DCGAN, CytoGAN, GAN-generated pseudo-macromolecular structures |

> ⚠️ **Important notes.** For research and educational purposes only, not to be used for clinical diagnosis.
> Every shape and image here is generated by `course_toolkit.simulate_nuclei_image`; no real microscopy data, published model weights, or external images are loaded.

## 🎯 Learning objectives

By the end of this notebook you will be able to:

- Explain the distinction between descriptive (discriminative) and generative models, and why some
  biological questions specifically require a generative model.
- Explain the geometric shape-space approach to generative modeling of cell/nuclear shape, and why naively
  interpolating pixel images does not produce valid new shapes.
- Explain a VAE's architecture (encoder, reparameterization trick, decoder) and its ELBO loss
  (reconstruction term + KL regularization term), and train a small VAE on simulated nucleus images.
- Explain the GAN minimax objective, the generator/discriminator roles, and the DCGAN convolutional
  architecture; train a minimal GAN on a toy 2-D distribution.
- Describe two real applications of adversarial generative modeling in bioimage analysis: CytoGAN
  (morphological cell profiling) and GAN-generated pseudo-macromolecular structures (cryo-electron
  tomography).

## 🗺️ Contents

1. Descriptive vs. generative models
2. Geometric shape-space models: distances, shape spaces, and why pixel interpolation fails
3. Variational autoencoders: architecture, the reparameterization trick, and the ELBO
4. Training a small VAE on simulated nucleus images
5. Generative adversarial networks: the minimax game and DCGAN
6. Applications: CytoGAN and GAN-generated pseudo-macromolecular structures

## 🧫 The biological question

A classifier (Module 07–08) can tell a round nucleus from an elongated one; it cannot tell you what an
*elongated nucleus looks like in general*, nor produce a brand-new, plausible example of one. Many
biological questions are exactly this second kind: what does normal cell morphology look like, as a
distribution, and how does it shift under a drug treatment or a genetic perturbation? Answering that well
requires a *generative* model — one that captures everything characteristic of a class, not just whatever
minimally distinguishes it from other classes.

---

## Environment setup

In [ ]:
# ── Hardware check ──
import platform
print('Python', platform.python_version())
try:
    import torch
    print('Torch', torch.__version__, '| CUDA available:', torch.cuda.is_available())
except ImportError:
    print('PyTorch not installed yet — the next cell installs it if needed.')


In [ ]:
!pip install -q scikit-image torch
# ── Dependencies ── (Colab already has most of these; we install only what is missing)


In [ ]:
# ── Global imports and configuration ──
import numpy as np
import matplotlib.pyplot as plt
from scipy import ndimage as ndi
import torch
import torch.nn as nn
import torch.nn.functional as F
torch.manual_seed(130)
SEED = 130
rng = np.random.default_rng(SEED)
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.titlesize'] = 11


### 🧰 Course toolkit

In [ ]:
# ── Course toolkit (identical to ../course_toolkit.py v1.0; copied so this notebook runs alone) ──
import numpy as np
import matplotlib.pyplot as plt
from scipy import ndimage as ndi
from scipy.optimize import linear_sum_assignment

SHAPE_CLASSES = ("round", "elongated", "kidney", "ring", "dividing", "irregular")


def _draw_blob(canvas: np.ndarray, cy: float, cx: float, r: float, shape: str,
                rng: np.random.Generator, angle: float = 0.0) -> None:
    """Add one synthetic nucleus/cell of the given shape class onto ``canvas`` (in place, additive)."""
    size = canvas.shape[0]
    yy, xx = np.meshgrid(np.arange(size), np.arange(size), indexing="ij")
    y, x = yy - cy, xx - cx
    ca, sa = np.cos(angle), np.sin(angle)
    yr, xr = ca * y + sa * x, -sa * y + ca * x  # rotate into the blob's own frame
    if shape == "round":
        d = (yr / r) ** 2 + (xr / r) ** 2
        canvas += np.exp(-d * 2.0)
    elif shape == "elongated":
        d = (yr / (r * 0.55)) ** 2 + (xr / (r * 1.7)) ** 2
        canvas += np.exp(-d * 2.0)
    elif shape == "kidney":
        d = (yr / r) ** 2 + (xr / r) ** 2
        dent = np.exp(-((yr - 0.0) ** 2 + (xr - r * 0.9) ** 2) / (2 * (r * 0.5) ** 2))
        canvas += np.clip(np.exp(-d * 2.0) - 0.9 * dent, 0, None)
    elif shape == "ring":
        d = np.sqrt(yr ** 2 + xr ** 2)
        canvas += np.exp(-((d - r) ** 2) / (2 * (r * 0.28) ** 2))
    elif shape == "dividing":
        sep = r * 1.1
        for sign in (-1, 1):
            d = ((yr - sign * sep * 0.5) / (r * 0.75)) ** 2 + (xr / (r * 0.75)) ** 2
            canvas += np.exp(-d * 2.0)
    else:  # irregular: a round blob perturbed by a few random bumps
        d = (yr / r) ** 2 + (xr / r) ** 2
        base = np.exp(-d * 2.0)
        for _ in range(3):
            bx, by = rng.uniform(-r, r, size=2)
            bump = np.exp(-(((yr - by) ** 2 + (xr - bx) ** 2) / (2 * (r * 0.45) ** 2)))
            base = np.clip(base + 0.5 * bump, 0, 1.3)
        canvas += base


def add_gaussian_noise(image: np.ndarray, sigma: float, rng: np.random.Generator) -> np.ndarray:
    """Additive Gaussian (read) noise: the classic noise floor of a camera sensor."""
    return image + rng.normal(0.0, sigma, size=image.shape)


def blur_image(image: np.ndarray, psf: np.ndarray) -> np.ndarray:
    """Convolve an image with a point-spread function (microscope optical blur)."""
    return ndi.convolve(image, psf, mode="reflect")


def gaussian_psf(sigma: float, radius: int | None = None) -> np.ndarray:
    """A normalised 2-D Gaussian point-spread function (the microscope's blur kernel)."""
    if radius is None:
        radius = max(1, int(np.ceil(3 * sigma)))
    ax = np.arange(-radius, radius + 1)
    xx, yy = np.meshgrid(ax, ax)
    k = np.exp(-(xx ** 2 + yy ** 2) / (2 * sigma ** 2))
    return k / k.sum()


def simulate_nuclei_image(n_cells: int = 25, size: int = 160, touching: bool = False,
                           shape_classes: tuple[str, ...] | None = None,
                           psf_sigma: float = 1.2, background: float = 0.05,
                           noise_sigma: float = 0.03, seed: int = 0):
    """A SIMULATED single-channel fluorescence image of cell nuclei/cells.

    Returns ``image`` (size x size, background + PSF-blurred blobs + Gaussian noise),
    ``instance_mask`` (0 = background, 1..n_cells = one integer id per cell, the ground truth
    for segmentation/detection/tracking), ``centroids`` (n_cells x 2, row, col), ``shape_labels``
    (one of SHAPE_CLASSES per cell, the ground truth for classification), and ``boxes`` (n_cells x 4,
    ymin, xmin, ymax, xmax — the ground truth for detection).
    """
    rng = np.random.default_rng(seed)
    classes = shape_classes or SHAPE_CLASSES
    image = np.full((size, size), background, dtype=float)
    instance_mask = np.zeros((size, size), dtype=int)
    centroids, labels, boxes = [], [], []
    margin = size * 0.12
    min_sep = size * 0.11 if not touching else size * 0.065
    placed = []
    attempts = 0
    while len(placed) < n_cells and attempts < n_cells * 60:
        attempts += 1
        cy, cx = rng.uniform(margin, size - margin, size=2)
        if all(np.hypot(cy - py, cx - px) > min_sep for py, px in placed):
            placed.append((cy, cx))
    for i, (cy, cx) in enumerate(placed):
        shape = classes[rng.integers(len(classes))]
        r = rng.uniform(size * 0.028, size * 0.045)
        angle = rng.uniform(0, 2 * np.pi)
        single = np.zeros_like(image)
        _draw_blob(single, cy, cx, r, shape, rng, angle)
        image += single
        this_mask = single > 0.25
        instance_mask[this_mask & (instance_mask == 0)] = i + 1
        centroids.append((cy, cx))
        labels.append(shape)
        ys, xs = np.where(this_mask)
        if len(ys):
            boxes.append((ys.min(), xs.min(), ys.max(), xs.max()))
        else:
            boxes.append((cy - r, cx - r, cy + r, cx + r))
    image = blur_image(image, gaussian_psf(psf_sigma))
    image = add_gaussian_noise(image, noise_sigma, rng)
    image = np.clip(image, 0, None)
    return dict(image=image, instance_mask=instance_mask, centroids=np.array(centroids),
                shape_labels=np.array(labels), boxes=np.array(boxes), source="SIMULATED")


def shape_features(mask: np.ndarray) -> dict:
    """Simple morphological/shape features of a binary blob mask: area, perimeter (boundary
    pixel count), equivalent-circle compactness, and the aspect ratio of its bounding box."""
    area = mask.sum()
    eroded = ndi.binary_erosion(mask)
    perimeter = max(1, (mask & ~eroded).sum())
    compactness = 4 * np.pi * area / (perimeter ** 2 + 1e-9)  # 1.0 for a perfect circle
    ys, xs = np.nonzero(mask)
    if len(ys) == 0:
        return dict(area=0, perimeter=0, compactness=0.0, aspect_ratio=1.0)
    h = ys.max() - ys.min() + 1
    w = xs.max() - xs.min() + 1
    aspect_ratio = max(h, w) / max(1, min(h, w))
    return dict(area=float(area), perimeter=float(perimeter),
                compactness=float(compactness), aspect_ratio=float(aspect_ratio))


def pca(X: np.ndarray, n_components: int = 2):
    """Principal component analysis by SVD of the column-centred data. Returns the projected
    scores (N x n_components), the components (n_components x J) and the explained-variance
    ratio of each component."""
    mean = X.mean(axis=0)
    Xc = X - mean
    U, S, Vt = np.linalg.svd(Xc, full_matrices=False)
    scores = (U * S)[:, :n_components]
    explained = (S ** 2) / np.sum(S ** 2)
    return scores, Vt[:n_components], explained[:n_components]


def show_image(ax, image: np.ndarray, title: str = "", cmap: str = "gray"):
    """Show one image on a given Matplotlib axis, titled, with ticks removed."""
    ax.imshow(image, cmap=cmap)
    ax.set_title(title, fontsize=10)
    ax.set_xticks([]); ax.set_yticks([])


def show_image_grid(images, titles=None, ncols: int = 4, cmap: str = "gray", figsize_scale: float = 2.6):
    """A grid of images with Matplotlib, one subplot each, titles optional."""
    n = len(images)
    ncols = min(ncols, n)
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(figsize_scale * ncols, figsize_scale * nrows))
    axes = np.atleast_1d(axes).ravel()
    for i, img in enumerate(images):
        show_image(axes[i], img, titles[i] if titles else "", cmap=cmap)
    for ax in axes[n:]:
        ax.axis("off")
    fig.tight_layout()
    return fig, axes

---

## 1. Descriptive vs. generative models

A **descriptive (discriminative) model** needs only enough information to tell classes apart: is this
protein at the ER or at nucleoli? Color or texture alone might already answer that, with no need to
represent everything else about what an ER-localized protein's image looks like. A **generative model**
instead asks: what does the *whole* distribution of images of this class look like? This needs a
*complete* description, able to produce (not just discriminate) instances that capture every characteristic
feature of the class — and, once we have that, we can do things no classifier can: generate novel, never-
seen examples; interpolate meaningfully between two examples; and notice when a new image doesn't fit the
learned distribution at all (anomaly/novelty detection).

Three generative strategies, in increasing order of how directly they model images, structure the rest of
this notebook: build an explicit geometric **shape space** from hand-designed shape descriptors (Section
2); learn a nonlinear, probabilistic latent space with a **variational autoencoder** (Sections 3–4); or
skip explicit density modeling entirely and train a network to produce samples a discriminator cannot tell
from real ones — a **generative adversarial network** (Sections 5–6).

## 2. Geometric shape-space models: distances, shape spaces, and why pixel
interpolation fails

A nonparametric way to represent a cell or nuclear shape is simply as a binary image (1 inside the shape,
0 outside) — this preserves full detail and every such image is automatically a *valid* shape. The trouble
is that binary shape images are poor material for building a generative *model*: fading linearly between
two shape images at the pixel level does not produce a third, recognizable shape — it produces a blurry,
semi-transparent overlay that isn't a valid binary shape at all. One established remedy,
**diffeomorphic shape analysis** (as used by CMU's CellOrganizer toolkit), instead measures the distance
between two shapes as the amount of smooth, invertible (diffeomorphic) deformation — registration "work" —
needed to morph one shape's boundary onto the other, and builds a shape space from those pairwise distances
via multidimensional scaling (Module 09), exactly as Module 09's MDS turned a city distance-matrix into
2-D coordinates. We will not implement full diffeomorphic registration here (it requires iterative
boundary-deformation algorithms beyond this notebook's scope); instead we demonstrate the pixel-
interpolation pitfall directly, then build a simplified, CellOrganizer-style shape space from PCA of
hand-designed shape *features* rather than raw pixels — descriptors that, unlike pixels, interpolate
sensibly.

In [ ]:
# ── 2.1 The pitfall: naive pixel-level interpolation between two shapes ──
shape_a = simulate_nuclei_image(n_cells=1, size=48, shape_classes=("round",), seed=11)
shape_b = simulate_nuclei_image(n_cells=1, size=48, shape_classes=("elongated",), seed=12)
mask_a = (shape_a["instance_mask"] > 0).astype(float)
mask_b = (shape_b["instance_mask"] > 0).astype(float)

fig, axes = plt.subplots(1, 5, figsize=(12, 2.6))
for ax, t in zip(axes, [0.0, 0.25, 0.5, 0.75, 1.0]):
    blended = (1 - t) * mask_a + t * mask_b
    ax.imshow(blended, cmap="gray", vmin=0, vmax=1)
    ax.set_title(f"t={t:.2f}"); ax.set_xticks([]); ax.set_yticks([])
fig.suptitle("Naive pixel averaging: not a valid shape at intermediate t", y=1.05)
plt.tight_layout(); plt.show()

🔍 **What just happened?** At $t=0$ and $t=1$ we see the two real shapes; at
every value in between, the "shape" is a semi-transparent gray blob with no crisp boundary at all — not a
valid binary shape, and not what either cell actually looks like partway through a transformation. This is
exactly the failure the lecture material warns about: pixels are the wrong representation to interpolate
in directly.

In [ ]:
# ── 2.2 Build a shape space from PCA of hand-designed shape features (not pixels) ──
shape_bank, feature_rows, class_rows = [], [], []
for cls in SHAPE_CLASSES:
    for seed in range(12):
        sim = simulate_nuclei_image(n_cells=1, size=48, shape_classes=(cls,), seed=1000 * SHAPE_CLASSES.index(cls) + seed)
        mask = sim["instance_mask"] > 0
        feats = shape_features(mask)
        shape_bank.append(mask)
        feature_rows.append([feats["compactness"], feats["aspect_ratio"],
                              feats["area"] / (48 * 48), feats["perimeter"] / 48])
        class_rows.append(cls)

feature_matrix = np.array(feature_rows)
feature_matrix = (feature_matrix - feature_matrix.mean(axis=0)) / feature_matrix.std(axis=0)
scores, components, explained = pca(feature_matrix, n_components=2)
print("explained variance ratio of the 2-D shape space:", np.round(explained, 3))

plt.figure(figsize=(6, 5))
for cls in SHAPE_CLASSES:
    sel = np.array(class_rows) == cls
    plt.scatter(scores[sel, 0], scores[sel, 1], label=cls, s=28, alpha=0.8)
plt.xlabel("PC1 (shape space)"); plt.ylabel("PC2 (shape space)")
plt.title("A CellOrganizer-style shape space (PCA of shape features)"); plt.legend(fontsize=8)
plt.tight_layout(); plt.show()

🧪 **Ground-truth check.** Because every shape's true class is known (the
simulator generated it), confirm visually that same-class points cluster together in this 2-D shape space
even though PCA was never told the class labels — exactly Module 09's PCA/eigenimage logic, applied here
to shape descriptors instead of raw pixel intensities.

In [ ]:
# ── 2.3 "Space to shapes": synthesize a shape at an arbitrary point via nearest neighbors ──
def synthesize_at_point(point, k=3):
    dists = np.linalg.norm(scores - point, axis=1)
    nearest = np.argsort(dists)[:k]
    return nearest, dists[nearest]

query_point = (scores[class_rows.index("round")] + scores[class_rows.index("elongated")]) / 2
nearest_idx, nearest_dist = synthesize_at_point(query_point, k=3)
print("query point:", np.round(query_point, 2))
print("nearest real shapes:", [(class_rows[i], round(d, 3)) for i, d in zip(nearest_idx, nearest_dist)])

fig, axes = plt.subplots(1, len(nearest_idx), figsize=(2.6 * len(nearest_idx), 2.8))
for ax, i in zip(np.atleast_1d(axes), nearest_idx):
    ax.imshow(shape_bank[i], cmap="gray"); ax.set_title(class_rows[i]); ax.set_xticks([]); ax.set_yticks([])
fig.suptitle("Nearest real shapes to an arbitrary shape-space query point", y=1.05)
plt.tight_layout(); plt.show()

🔍 **What just happened?** Given an arbitrary point in the shape space (here,
the midpoint between a "round" and an "elongated" example), we *retrieved* the nearest real shapes rather
than truly generating a new one — a legitimate, classical "space to shapes" strategy, but notice it can
only ever return shapes already in the training bank. A model that could instead *decode* an arbitrary
point — including ones far from every training example — into a brand-new, valid image would be strictly
more powerful. That is precisely what a variational autoencoder's decoder does, which is why Sections 3–4
return to this same idea with a learned nonlinear decoder in place of nearest-neighbor retrieval.

### 📘 Lecture notes — CellOrganizer and diffeomorphic registration

CMU's [CellOrganizer](http://cellorganizer.org) builds generative models of cell and nuclear shape, and of
the spatial distribution of subcellular structures and proteins within them, directly from microscopy
images. Its shape component uses true diffeomorphic registration distances (not the feature-based proxy
used in Section 2.2) between real nuclear/cell outlines, so that the resulting shape space reflects actual
deformation cost rather than a hand-chosen feature summary — more faithful, at the cost of a substantially
more expensive distance computation between every pair of shapes.

## 3. Variational autoencoders: architecture, the reparameterization trick,
and the ELBO

A **variational autoencoder (VAE)** is a probabilistic reinterpretation of the plain autoencoder from
Module 09. Where a plain autoencoder's encoder outputs one deterministic latent vector $z$, a VAE's
encoder outputs the *parameters of a distribution* over $z$ — a mean $\mu(x)$ and a standard deviation
$\sigma(x)$ for an (assumed independent, per-dimension) Gaussian — and then *samples* $z \sim
\mathcal{N}(\mu(x), \sigma(x)^2)$ before decoding. This makes the latent space a genuine probability
distribution we can later sample from to generate new data, not just a deterministic bottleneck.

**The reparameterization trick.** Sampling $z \sim \mathcal{N}(\mu, \sigma^2)$ directly is a random,
non-differentiable operation — gradients cannot flow back through it to update the encoder. The fix:
rewrite the sample as $z = \mu + \sigma \odot \epsilon$, where $\epsilon \sim \mathcal{N}(0, 1)$ is
drawn *first* and treated as a fixed input, not a parameter. Now $z$ is a deterministic, differentiable
function of $\mu$, $\sigma$, and $\epsilon$, and ordinary backpropagation works.

**The ELBO (evidence lower bound) loss.** Maximizing the true data likelihood $p(x) = \int p(z)p(x|z)\,dz$
directly is intractable (it requires integrating over every possible $z$). The ELBO is a tractable
surrogate, maximized instead:

$$\mathcal{L} = \underbrace{\mathbb{E}_{z \sim q(z|x)}\big[\log p(x|z)\big]}_{\text{reconstruction term}}
\;-\; \underbrace{D_{\mathrm{KL}}\big(q(z|x)\,\|\,p(z)\big)}_{\text{regularization term}}$$

The reconstruction term pushes decoded $\hat{x}$ to look like the input $x$ (in practice, often just
mean-squared-error for continuous $x$). The regularization term pulls each image's encoded distribution
$q(z|x)$ toward the chosen prior $p(z)$ (conventionally standard Gaussian, $\mathcal{N}(0, 1)$) — without
it, the encoder could cheat by scattering different inputs across disconnected, far-apart regions of latent
space with near-zero variance, which would reconstruct perfectly but leave huge "holes" that decode to
nothing meaningful, making the latent space useless for sampling new points.

In [ ]:
# ── 3.1 The reparameterization trick: gradients flow through mu and sigma, not the sample ──
mu = torch.tensor([0.5, -1.2], requires_grad=True)
log_sigma = torch.tensor([0.1, -0.3], requires_grad=True)
sigma = torch.exp(log_sigma)

epsilon = torch.randn_like(mu)          # sampled first, then held fixed -- NOT a learned parameter
z = mu + sigma * epsilon                # a deterministic function of mu, sigma (and epsilon)

dummy_loss = (z ** 2).sum()
dummy_loss.backward()
print("gradient reached mu     :", mu.grad)
print("gradient reached sigma  :", sigma.grad if sigma.requires_grad else "(sigma is a derived tensor)")
print("gradient reached log_sigma:", log_sigma.grad)

🔍 **What just happened?** Both `mu` and `log_sigma` received real gradients
from a loss computed on the *sampled* `z` — exactly what makes end-to-end training possible. Without the
reparameterization trick (sampling `z` directly from a `Normal(mu, sigma)` distribution object with no
`epsilon` separated out), `z` would not be a differentiable function of the encoder's outputs, and
`mu.grad`/`log_sigma.grad` would be `None`.

## 4. Training a small VAE on simulated nucleus images

We now train a compact convolutional VAE on $32\times32$ crops of simulated nuclei spanning all six shape
classes, with a 2-D latent space (chosen small and plottable specifically so we can visualize it directly,
as Module 09 did for its plain autoencoder).

In [ ]:
# ── 4.1 Build a training set of single-nucleus image crops across all shape classes ──
def make_vae_dataset(n_per_class=60, size=32, seed0=0):
    images, labels = [], []
    for ci, cls in enumerate(SHAPE_CLASSES):
        for k in range(n_per_class):
            sim = simulate_nuclei_image(n_cells=1, size=size, shape_classes=(cls,),
                                         noise_sigma=rng.uniform(0.02, 0.04), seed=10_000 * ci + k + seed0)
            images.append(sim["image"])
            labels.append(cls)
    X = torch.from_numpy(np.stack(images)[:, None]).float()
    return X, labels

X_vae, labels_vae = make_vae_dataset(n_per_class=50, seed0=2000)
print("training set:", X_vae.shape)

In [ ]:
# ── 4.2 A small convolutional VAE with a 2-D latent space ──
class SmallVAE(nn.Module):
    def __init__(self, latent_dim=2):
        super().__init__()
        self.enc_conv = nn.Sequential(
            nn.Conv2d(1, 16, 4, stride=2, padding=1), nn.ReLU(),    # 32 -> 16
            nn.Conv2d(16, 32, 4, stride=2, padding=1), nn.ReLU(),   # 16 -> 8
        )
        self.fc_mu = nn.Linear(32 * 8 * 8, latent_dim)
        self.fc_logvar = nn.Linear(32 * 8 * 8, latent_dim)
        self.fc_dec = nn.Linear(latent_dim, 32 * 8 * 8)
        self.dec_conv = nn.Sequential(
            nn.ConvTranspose2d(32, 16, 4, stride=2, padding=1), nn.ReLU(),   # 8 -> 16
            nn.ConvTranspose2d(16, 1, 4, stride=2, padding=1),               # 16 -> 32
        )

    def encode(self, x):
        h = self.enc_conv(x).flatten(1)
        return self.fc_mu(h), self.fc_logvar(h)

    def reparameterize(self, mu, logvar):
        sigma = torch.exp(0.5 * logvar)
        return mu + sigma * torch.randn_like(sigma)

    def decode(self, z):
        h = self.fc_dec(z).view(-1, 32, 8, 8)
        return torch.sigmoid(self.dec_conv(h))

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        return self.decode(z), mu, logvar

def vae_loss(x_hat, x, mu, logvar):
    recon = F.mse_loss(x_hat, x, reduction="sum") / x.shape[0]
    kl = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp()) / x.shape[0]
    return recon + kl, recon, kl

vae = SmallVAE(latent_dim=2)
optimizer = torch.optim.Adam(vae.parameters(), lr=1e-3)
print(vae)

In [ ]:
# ── 4.3 Train ──
history = []
for epoch in range(40):
    perm = torch.randperm(len(X_vae))
    epoch_loss = epoch_recon = epoch_kl = 0.0
    for i in range(0, len(X_vae), 32):
        idx = perm[i:i + 32]
        optimizer.zero_grad()
        x_hat, mu, logvar = vae(X_vae[idx])
        loss, recon, kl = vae_loss(x_hat, X_vae[idx], mu, logvar)
        loss.backward(); optimizer.step()
        epoch_loss += loss.item() * len(idx); epoch_recon += recon.item() * len(idx); epoch_kl += kl.item() * len(idx)
    n = len(X_vae)
    history.append((epoch_loss / n, epoch_recon / n, epoch_kl / n))

history = np.array(history)
plt.plot(history[:, 0], label="total (ELBO) loss")
plt.plot(history[:, 1], label="reconstruction term")
plt.plot(history[:, 2], label="KL term")
plt.xlabel("epoch"); plt.ylabel("loss"); plt.legend(); plt.title("VAE training"); plt.tight_layout(); plt.show()

In [ ]:
# ── 4.4 The learned 2-D latent space, colored by (never-trained-on) true shape class ──
vae.eval()
with torch.no_grad():
    mu_all, _ = vae.encode(X_vae)
mu_all = mu_all.numpy()

plt.figure(figsize=(6, 5))
for cls in SHAPE_CLASSES:
    sel = np.array(labels_vae) == cls
    plt.scatter(mu_all[sel, 0], mu_all[sel, 1], s=14, alpha=0.7, label=cls)
plt.xlabel("latent dim 1"); plt.ylabel("latent dim 2")
plt.title("VAE latent space (unsupervised), colored by true class"); plt.legend(fontsize=8)
plt.tight_layout(); plt.show()

🧪 **Ground-truth check.** As in Module 09's autoencoder, the VAE was never
given class labels during training — confirm that same-class points nonetheless group together, meaning
the unsupervised reconstruction + KL objective alone discovered structure aligned with true shape class.

In [ ]:
# ── 4.5 Generate brand-new images by sampling the prior, and interpolate in latent space ──
with torch.no_grad():
    z_samples = torch.randn(8, 2)                    # sample directly from the prior p(z) = N(0, 1)
    generated = vae.decode(z_samples).numpy()[:, 0]
show_image_grid(list(generated), titles=[f"sample {i}" for i in range(8)], ncols=4)
plt.suptitle("Brand-new images decoded from random prior samples (never in the training set)", y=1.02)
plt.show()

with torch.no_grad():
    mu_round, _ = vae.encode(X_vae[np.array(labels_vae) == "round"][:1])
    mu_elong, _ = vae.encode(X_vae[np.array(labels_vae) == "elongated"][:1])
    ts = torch.linspace(0, 1, 7).unsqueeze(1)
    z_path = (1 - ts) * mu_round + ts * mu_elong
    interp_images = vae.decode(z_path).numpy()[:, 0]
show_image_grid(list(interp_images), titles=[f"t={t:.2f}" for t in np.linspace(0, 1, 7)], ncols=7)
plt.suptitle("Latent-space interpolation: round -> elongated (contrast with Section 2.1's pixel blend)", y=1.05)
plt.show()

🔍 **What just happened?** Unlike Section 2.1's naive pixel averaging, every
frame along this latent-space path is decoded through the trained network, so each intermediate image is a
plausible, crisp shape, not a fading overlay — the decoder has learned a *nonlinear* manifold on which
"straight lines" correspond to smooth, valid morphs, which is exactly what a shape space built from true
diffeomorphic distances (Section 2's lecture note) is designed to achieve, but learned directly from pixel
data rather than hand-designed shape descriptors.

✏️ **Your turn.** Repeat Section 4.5's interpolation between two *different*
classes of your choosing (e.g., "round" to "ring", or "kidney" to "irregular"). Does every intermediate
frame still look like a single, plausible, if ambiguous, nucleus shape, or do some interpolation paths pass
through a frame that looks implausible? (Both outcomes are informative about how well the latent space has
organized that particular pair of classes.)

### 📘 Lecture notes — latent perturbation and disentanglement

**Latent perturbation**: holding every other latent dimension fixed and slowly sweeping just one reveals
what that dimension has learned to encode — in a well-organized latent space, different dimensions capture
different, interpretable factors of variation (e.g., one dimension might track overall size, another
elongation). **Disentanglement** is the (not automatically guaranteed) property that these dimensions vary
*independently* of one another; it is encouraged by the same diagonal, independent Gaussian prior
$p(z) = \mathcal{N}(0, I)$ already used in Section 3's KL term — strengthening that regularization term's
weight (as in the $\beta$-VAE variant) trades off reconstruction sharpness for more disentangled, more
interpretable latent dimensions. Real applications of VAEs to cell image profiling — learning
representations whose structure captures cell-population variation under different drug treatments — use
exactly this latent perturbation idea to interpret what a trained model has learned, and VAE-based
representations have been reported to identify a compound's mechanism-of-action (e.g., correctly
identifying a microtubule-destabilizing drug) in ways plain autoencoder representations missed.

## 5. Generative adversarial networks: the minimax game and DCGAN

A **generative adversarial network (GAN)** takes a completely different approach: rather than explicitly
modeling a density (as VAEs do, via the ELBO), train two competing networks. A **generator** $G$ maps
random noise $z$ to a fake sample $G(z)$, trying to fool a **discriminator** $D$; the discriminator tries
to correctly tell real training samples from the generator's fakes. Formally, this is a minimax game:

$$\min_G \max_D \; \mathbb{E}_{x \sim p_{\text{data}}}\big[\log D(x)\big] +
\mathbb{E}_{z \sim p(z)}\big[\log(1 - D(G(z)))\big]$$

$D$ wants $D(x)\to1$ on real data and $D(G(z))\to0$ on fakes (maximizing the objective); $G$ wants
$D(G(z))\to1$ — fooling $D$ into thinking its fakes are real (minimizing the objective). Neither network
ever needs an explicit formula for $p_{\text{data}}(x)$ — the GAN learns only to *sample* convincingly,
which is why it is called an **implicit density model** (no KL divergence or explicit likelihood anywhere
in its training, unlike a VAE).

**DCGAN** (deep convolutional GAN) specifies a concrete, now-standard convolutional architecture: the
generator is an upsampling network built from transposed convolutions (Module 11's `ConvTranspose2d`,
mapping low-dimensional noise up to a full image), and the discriminator is an ordinary convolutional
classifier (strided convolutions downsampling toward a single real/fake score) — no pooling or
fully-connected layers in either network, which DCGAN's authors found gave substantially more stable
training than earlier GAN architectures.

In [ ]:
# ── 5.1 A minimal GAN trained on a toy 2-D distribution: Section 2's shape-feature space ──
real_points = torch.from_numpy(scores).float()                 # the PCA shape-space coordinates from Sec. 2.2
real_points = (real_points - real_points.mean(0)) / real_points.std(0)

class TinyGenerator(nn.Module):
    def __init__(self, noise_dim=4):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(noise_dim, 32), nn.ReLU(), nn.Linear(32, 32), nn.ReLU(), nn.Linear(32, 2))
    def forward(self, z): return self.net(z)

class TinyDiscriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(2, 32), nn.ReLU(), nn.Linear(32, 32), nn.ReLU(), nn.Linear(32, 1))
    def forward(self, x): return self.net(x)

G, D = TinyGenerator(), TinyDiscriminator()
opt_G = torch.optim.Adam(G.parameters(), lr=2e-3)
opt_D = torch.optim.Adam(D.parameters(), lr=2e-3)
bce = nn.BCEWithLogitsLoss()
NOISE_DIM = 4

for epoch in range(400):
    idx = torch.randint(0, len(real_points), (64,))
    real_batch = real_points[idx]
    noise = torch.randn(64, NOISE_DIM)
    fake_batch = G(noise)

    opt_D.zero_grad()
    d_loss = bce(D(real_batch), torch.ones(64, 1)) + bce(D(fake_batch.detach()), torch.zeros(64, 1))
    d_loss.backward(); opt_D.step()

    opt_G.zero_grad()
    g_loss = bce(D(fake_batch), torch.ones(64, 1))   # generator wants D(fake) -> 1
    g_loss.backward(); opt_G.step()

with torch.no_grad():
    final_fake = G(torch.randn(200, NOISE_DIM)).numpy()
plt.figure(figsize=(6, 5))
plt.scatter(real_points[:, 0], real_points[:, 1], s=20, alpha=0.5, label="real (shape-space points)")
plt.scatter(final_fake[:, 0], final_fake[:, 1], s=20, alpha=0.5, label="generator samples")
plt.legend(); plt.title("A minimal GAN learning to imitate Section 2's shape space")
plt.tight_layout(); plt.show()

🔍 **What just happened?** `G`, a small MLP, started by mapping random noise
to an arbitrary, unrelated cloud of points; after 400 rounds of the adversarial game, its samples (orange)
should visibly overlap the real shape-space distribution (blue) — with no density function, reconstruction
loss, or KL term anywhere in the training loop, only the discriminator's binary real/fake signal. This toy
2-D case uses plain fully-connected layers; DCGAN replaces them with the convolutional/transposed-
convolutional layers described above specifically to scale this same minimax idea up to realistic images.

✏️ **Your turn.** Increase `NOISE_DIM` from 4 to 1 and retrain. With only a
single noise dimension feeding a 2-D output space, can the generator still cover the full real
distribution, or does it collapse onto a lower-dimensional curve? (This is a small-scale preview of **mode
collapse**, a well-known GAN training failure mode where the generator stops covering the full diversity of
the real distribution.)

## 6. Applications: CytoGAN and GAN-generated pseudo-macromolecular structures

**CytoGAN** (Goldsborough et al., 2017) applies a DCGAN-style adversarial framework to fluorescence
microscopy images of cultured human cells, for *morphological profiling*: mapping cell images to
representation vectors that capture the salient axes of morphological variation, in a fully unsupervised
way (no manual feature engineering, unlike Module 07's classical features). The paper reports that
adversarially-learned representations outperformed plain-autoencoder representations at grouping cells by
which drug treatment they received — and, because the model is generative, it can also visualize
interpolations between cell images and perform simple vector arithmetic on the representation space,
directly connecting the learned representation back to interpretable morphological variation (echoing
Section 3's latent-perturbation discussion).

**GAN-generated pseudo-macromolecular structures** (Wang et al., 2018) tackles a very different imaging
modality: cryo-electron tomography (cryo-ET), whose images have extremely low signal-to-noise ratio and a
"missing wedge" of unobserved angles, making macromolecule identification by template matching
statistically fragile (a simple correlation threshold against one template is not statistically rigorous).
Rigorous hypothesis testing needs a *null distribution* of what a non-matching macromolecule's appearance
could plausibly look like — but real macromolecular structures are scarce. The paper trains a 3-D GAN (a
direct extension of DCGAN's 2-D convolutional generator/discriminator to 3-D convolutions) on a modest
set of 15 known structures (augmented to 9000 by rotation), producing synthetic "pseudo-macromolecular"
volumes statistically similar to, but distinct from, the real training structures — populating exactly the
null distribution rigorous hypothesis testing needs, and demonstrating (as Section 5's toy example did)
that interpolating between two points in the GAN's learned latent space produces a smooth, continuous
morph between macromolecular shapes, computed far faster on a GPU than full diffeomorphic registration
(Section 2) between the same two shapes would be.

🧠 Both applications follow the same pattern as Section 5.1's toy demonstration
— a generator and discriminator playing the same minimax game — just with convolutional (CytoGAN, 2-D) or
3-D convolutional (pseudo-macromolecular structures) architectures in place of the small MLPs used here,
and with real microscopy or tomography images in place of Section 2's shape-feature coordinates.

📌 **Key takeaways**

- A generative model aims for a *complete* description of a class's distribution (able to produce new, characteristic instances), where a descriptive/discriminative model needs only enough information to tell classes apart.
- Pixel images do not interpolate meaningfully by simple averaging; diffeomorphic shape-space models instead measure shape distance by deformation cost, while this notebook's simplified, CellOrganizer-style stand-in uses PCA over hand-designed shape features.
- A VAE's encoder outputs a distribution's parameters ($\mu$, $\sigma$), not a single vector; the reparameterization trick ($z = \mu + \sigma\odot\epsilon$) makes sampling differentiable; the ELBO loss balances reconstruction accuracy against a KL term pulling the latent distribution toward a simple (typically standard Gaussian) prior.
- A GAN trains a generator and discriminator against each other in a minimax game, learning to *sample* realistically without ever modeling an explicit density; DCGAN specifies a stable convolutional/transposed-convolutional architecture for this.
- CytoGAN and GAN-generated pseudo-macromolecular structures show the same adversarial idea applied to real morphological profiling and cryo-ET hypothesis testing, respectively.

✅ **Checkpoint**

1. Why does the reparameterization trick rewrite $z \sim \mathcal{N}(\mu, \sigma^2)$ as
   $z = \mu + \sigma \odot \epsilon$ with $\epsilon \sim \mathcal{N}(0,1)$, rather than sampling $z$
   directly from that normal distribution?
   <details><summary>Answer</summary>Because direct sampling is a non-differentiable, random operation that
   gradients cannot flow back through; rewriting it isolates the randomness into a fixed input $\epsilon$,
   making $z$ a deterministic, differentiable function of the encoder's outputs $\mu$ and $\sigma$, so
   ordinary backpropagation can train the encoder.</details>
2. In the ELBO loss, what would likely happen to the learned latent space if the KL regularization term's
   weight were set to zero?
   <details><summary>Answer</summary>The encoder would have no incentive to keep $q(z|x)$ close to the
   prior, so different inputs could be mapped to disconnected, far-apart, very narrow regions of latent
   space (minimizing reconstruction loss without regard to the resulting latent geometry) — reconstruction
   could still look good, but sampling the prior or interpolating between points would likely land in
   "holes" that decode to meaningless images.</details>
3. Why is a GAN called an "implicit density model," and what does the discriminator's role have to do
   with that name?
   <details><summary>Answer</summary>A GAN never writes down or optimizes an explicit formula for
   $p_{\text{data}}(x)$ (no KL divergence or likelihood term, unlike a VAE's ELBO) — it only learns a
   function ($G$) that *samples* from something statistically close to the real distribution, judged
   entirely by whether the discriminator can still tell its samples from real data.</details>

🧠 **Questions & discussion**

- Section 2 retrieved nearest-neighbor real shapes rather than truly generating new pixels; Section 4's VAE decodes genuinely novel images instead. What would you lose, in terms of guarantees about validity, by using a VAE's decoder output versus returning an existing training example?
- A GAN has no reconstruction loss and no explicit encoder (Section 5's `G` only ever maps noise *forward* to an image). Given that, how would you check, for a specific real cell image, what latent code a trained GAN's generator would need to reproduce it? Is this even well-defined, the way it is for a VAE's encoder?
- CytoGAN's abstract reports that adversarially-learned representations were *superior* to autoencoder-based ones for grouping cells by treatment, yet *inferior* to classical computer vision and transfer-learning features for the same task at the time of publication. What does this suggest about when a fully unsupervised generative representation is worth using over hand-designed (Module 07) or supervised deep (Module 08) features?

🚀 **Bonus — going further**

- Increase `SmallVAE`'s `latent_dim` from 2 to 8 in Section 4.2, retrain, and compare reconstruction quality (plot a few real vs. reconstructed images side by side) against the 2-D version — does a higher-dimensional latent space reconstruct more faithfully, at the cost of being harder to visualize directly?
- In Section 5.1, replace the toy 2-D real distribution with Section 4's VAE latent means (`mu_all`) instead of the shape-feature PCA scores, and see whether the GAN can learn to imitate a *learned* latent distribution as well as a hand-designed feature-space one.
- Implement a simple `beta`-weighted ELBO (multiply the KL term in `vae_loss` by a `beta > 1` factor) and compare the resulting latent space's organization by class to the unweighted version from Section 4.4 — does a larger `beta` produce a more class-separated 2-D latent space, at the cost of blurrier reconstructions?

📝 **Lab note**

```text
Module 13 — Image synthesis and generative models
Date: ____________
VAE final ELBO / reconstruction / KL: ____ / ____ / ____
Latent space: class clusters visible?  yes / no   Which classes overlapped most? ____________
Toy GAN: did generator samples visibly cover the real distribution after training? yes / no
One real bioimage question I would want a generative (not just descriptive) model for: ____________
```

📚 **References**

- Johnson, G. R., Donovan-Maiye, R. M. & Maleckar, M. M. (2017). Generative modeling with conditional autoencoders: Building an integrated cell. *arXiv:1705.00092* (CellOrganizer-related work).
- Peng, T., Balestra, C., Murphy, R. F. et al. (2009). From the cover: Determining the distribution of probes between different subcellular locations through automated unmixing of subcellular patterns. *PNAS*.
- Kingma, D. P. & Welling, M. (2014). Auto-encoding variational Bayes. *ICLR 2014*.
- Goodfellow, I. et al. (2014). Generative adversarial nets. *NeurIPS 2014*.
- Radford, A., Metz, L. & Chintala, S. (2016). Unsupervised representation learning with deep convolutional generative adversarial networks (DCGAN). *ICLR 2016*.
- Goldsborough, P., Pawlowski, N., Caicedo, J. C., Singh, S. & Carpenter, A. (2017). CytoGAN: Generative modeling of cell images. *bioRxiv:227645*.
- Wang, K. W. et al. (2018). Image-derived generative modeling of pseudo-macromolecular structures — towards statistical assessment of electron cryotomography template matching. *BMVC 2018*.

➡️ **Next.** Module 14 turns from single still images to image *sequences*: why detecting objects frame by frame isn't the same problem as tracking them through time, and the data-association algorithms that connect detections across frames into trajectories.